# One transformer autoencoder run, end to end

This notebook runs a single **transformer autoencoder** experiment step by step, with a plot at every step, for one case:

**single frames → simulated data → observed flux stored in the file (no stellar spectra) → Gaussian noise added**

1. settings and config
2. data: load, split, inspect (observed flux, target tellurics, metadata, noise)
3. model: build, inspect, untrained prediction
4. training
5. loss curves
6. evaluation on the test nights, and the telluric-corrected spectra

**Model:** one observation in, one telluric spectrum out —
`observed (B, 2048) + wavelength (B, 2048) + metadata [airmass, PWV] (B, 2) → telluric (B, 2048)`: the spectrum is cut into patches (tokens) with the wavelengths as positional encoding, plus one metadata token; a transformer encoder compresses them into a few latent tokens, and a decoder with wavelength-encoded queries rebuilds the 2048 pixels.
Trained with MSE on the telluric spectrum (HDF5 dataset `flux`, set by `data.target_key`).

**Your data file** (`data.night_h5` in `experiments/single_frame/experiment_transformer_ae.yaml`, relative to `$TELLURICS_DATA`) needs:
the telluric target `flux`, the observed flux under `OBSERVED_KEY` (same shape), `wavelength`, and `labels` with the columns
`series_index` (flat layout), `airmass` and `PWV`. No PHOENIX files are used.
To try the notebook without any data, set `USE_SYNTHETIC = True` below: it builds a small synthetic file in a temporary folder.

## 1. Settings

In [ ]:
from pathlib import Path
import os, sys, tempfile

import numpy as np
import torch
import matplotlib.pyplot as plt

# The notebook lives in tests/: find the repo root and make the src-layout package importable.
_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd if (_cwd / "experiments").exists() else _cwd.parent
for p in (REPO_ROOT / "src", REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# ---- settings ------------------------------------------------------------- #
CONFIG = REPO_ROOT / "experiments" / "single_frame" / "experiment_transformer_ae.yaml"
OBSERVED_KEY = "observed"  # HDF5 dataset with the observed flux (the model input)
USE_SYNTHETIC = False      # True: build a small synthetic dataset (no real data needed)
MAX_EPOCHS = 5             # short notebook run; the YAML's max_epochs is for full runs
DEVICE = "auto"            # "auto" | "cpu" | "gpu"
OVERRIDES = []             # extra dotted overrides, e.g. ["model.latent_dim=64", "data.noise_level_mean=0.02"]
RUN_DIR = REPO_ROOT / "runs" / "notebook_transformer_ae"

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
torch.manual_seed(0); np.random.seed(0)
print("repo root:", REPO_ROOT)
print("config   :", CONFIG.relative_to(REPO_ROOT))

### 1b. (Optional) synthetic data

Only runs with `USE_SYNTHETIC = True`. It writes `sim.h5` in a temporary folder — flat layout (one row per observation,
nights with different numbers of frames): `flux` (the telluric target), `observed` (the stored observed flux: the tellurics
times a smooth continuum with a few lines), `wavelength`, labels `series_index`, `airmass`, `PWV` — and points the config at it.
The demo trains for 15 epochs with `noise_repeats = 2` at learning rate 1e-3 (a few minutes on a CPU). For speed on a CPU the synthetic demo also uses a smaller transformer (`d_model 64`, 2 + 2 layers); your YAML settings are used otherwise.

In [ ]:
if USE_SYNTHETIC:
    import h5py

    def write_synthetic(path, n_nights, rng, n_wave=2048):
        wl = np.linspace(2000.0, 2100.0, n_wave)                      # nm
        centres = rng.uniform(2000, 2100, 60)                          # telluric line positions (fixed)
        strengths = rng.uniform(0.2, 1.0, 60)
        profile = sum(s * np.exp(-0.5 * ((wl - c) / 0.04) ** 2) for c, s in zip(centres, strengths))
        rows, series = [], []
        for night in range(n_nights):
            pwv = rng.uniform(1, 20)                                   # mm, fixed per night
            continuum = 1 + 0.1 * np.sin(wl / rng.uniform(5, 15)) - 0.3 * np.exp(-0.5 * ((wl - rng.uniform(2000, 2100)) / 0.3) ** 2)
            for k in range(rng.integers(5, 15)):                       # unequal frames per night
                airmass = 1.0 + 0.9 * k / 15
                tell = np.exp(-0.08 * airmass * pwv * profile)
                rows.append((tell, tell * continuum, airmass, pwv)); series.append(night)
        tell, obs, airmass, pwv = (np.array(x) for x in zip(*rows))
        labels = np.column_stack([series, airmass, pwv]).astype(np.float32)
        with h5py.File(path, "w") as f:
            f.create_dataset("flux", data=tell.astype(np.float32))              # target
            f.create_dataset(OBSERVED_KEY, data=obs.astype(np.float32))         # model input (stored)
            f.create_dataset("wavelength", data=wl)
            f.create_dataset("labels", data=labels).attrs["columns"] = np.array(["series_index", "airmass", "PWV"], dtype="S")

    syn_root = Path(tempfile.mkdtemp(prefix="tellurics_synthetic_"))
    write_synthetic(syn_root / "sim.h5", n_nights=60, rng=np.random.default_rng(1))
    os.environ["TELLURICS_DATA"] = str(syn_root)
    MAX_EPOCHS = 15
    OVERRIDES = [
        "training.optimizer.learning_rate=1e-3", "data.noise_repeats=2",
        "data.night_h5=sim.h5", "data.max_nights=null", "model.d_model=64", "model.num_heads=4", "model.num_encoder_layers=2", "model.num_decoder_layers=2", "model.dim_feedforward=128", 
        *OVERRIDES,
    ]
    print("synthetic data in", syn_root)

## 2. Config

The YAML is loaded and this notebook's case is enforced on top of it: the input is the stored observed flux
(`data.observed_key`, so no stellar spectra), the data is simulated (so it gets noise), and no real observations are used.

In [ ]:
from tellurics.config import load_config
from tellurics.scripts.train import apply_overrides

CASE = [f"data.observed_key={OBSERVED_KEY}", "data.simulated=true", "data.real=[]"]
config = apply_overrides(load_config(CONFIG), [*CASE, *OVERRIDES, f"training.max_epochs={MAX_EPOCHS}"])
d, m, t = config.data, config.model, config.training
assert d.noise_level_mean > 0, "this notebook adds noise: set data.noise_level_mean > 0 in the YAML"

print(f"architecture : {m.architecture.value}  (N = {m.num_wavelength_bins})")
print(f"data file    : {d.night_h5}")
print(f"input        : stored observed flux {d.observed_key!r} (no stellar spectra)")
print(f"target       : HDF5 dataset {d.target_key!r}")
print(f"noise        : level ~ N({d.noise_level_mean:.2%}, {d.noise_level_std:.2%}) of each pixel per observation; "
      f"train/val repeated x{d.noise_repeats} per epoch")
print(f"metadata     : {d.metadata_columns}  bounds {d.bounds}")
print(f"training     : {t.max_epochs} epochs, batch {t.batch_size}, lr {t.optimizer.learning_rate}")

## 3. Data

The data module splits **by night** (no frame of a validation/test night is seen in training) and adds the Gaussian noise
to the inputs: fresh in training, fixed in validation/test. Every observation draws its own noise level from
N(`noise_level_mean`, `noise_level_std`²); with `noise_repeats = R` the training and validation sets are served R times per
epoch, each time with a different noise realization (augmentation).

In [ ]:
from tellurics.data.datamodule import TelluricDataModule

RUN_DIR.mkdir(parents=True, exist_ok=True)
dm = TelluricDataModule(d, run_dir=RUN_DIR, batch_size=t.batch_size,
                        num_workers=0, single_frame=m.single_frame)
dm.setup()
assert dm.stellar_pool is None                      # no stellar spectra in this case
wl = dm.wavegrid.wavelength
for name, ds, ids in (("train", dm.train_ds, dm.train_ids), ("val", dm.val_ds, dm.val_ids), ("test", dm.test_ds, dm.test_ids)):
    print(f"{name:5s}: {len(ids):4d} nights -> {ds._n_base():6d} observations "
          f"({len(ds)} samples per epoch with x{ds.repeats} repeats)")

### 3a. One observation: input vs target

In [ ]:
sample = dm.val_ds[0]                        # validation: fixed noise
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(wl, sample["observed"], lw=0.7, label=f"observed (model input, noise level {float(sample['noise_level']):.2%})")
axes[0].set_ylabel("flux"); axes[0].legend(loc="lower left")
axes[1].plot(wl, sample["telluric"], lw=0.7, color="C2", label="telluric spectrum (target, 'flux')")
axes[1].set_ylabel("transmission"); axes[1].set_xlabel(f"wavelength [{dm.wavegrid.unit}]"); axes[1].legend(loc="lower left")
axes[0].set_title(f"night {sample['night_id']}, frame {sample['frame_id']}  |  metadata (scaled) = {sample['theta']['metadata'].numpy().round(3)}")
plt.tight_layout(); plt.show()

### 3b. Metadata per split and frames per night

In [ ]:
def metadata_of(ds, n=2000):
    idx = np.linspace(0, ds._n_base() - 1, min(n, ds._n_base())).astype(int)
    return np.stack([ds[i]["theta"]["metadata"].numpy() for i in idx])

cols = d.metadata_columns
fig, axes = plt.subplots(1, len(cols) + 1, figsize=(4.2 * (len(cols) + 1), 3.3))
for name, ds in (("train", dm.train_ds), ("val", dm.val_ds), ("test", dm.test_ds)):
    meta = metadata_of(ds)
    for j, c in enumerate(cols):
        axes[j].hist(meta[:, j], bins=30, alpha=0.5, density=True, label=name)
for j, c in enumerate(cols):
    axes[j].set_title(f"{c} (scaled to [0, 1])"); axes[j].legend()
if dm.night_rows is not None:
    counts = [len(r) for r in dm.night_rows.values()]
    axes[-1].hist(counts, bins=np.arange(min(counts), max(counts) + 2) - 0.5)
    axes[-1].set_title("frames per night"); axes[-1].set_xlabel("frames")
else:
    axes[-1].text(0.5, 0.5, f"{dm.n_frames} frames every night", ha="center"); axes[-1].set_axis_off()
plt.tight_layout(); plt.show()

### 3c. The input noise

In [ ]:
ds = dm.train_ds
n = min(64, ds._n_base())
samples = [ds[i] for i in range(n)]
noisy = np.stack([x["observed"].numpy() for x in samples])
levels = np.array([float(x["noise_level"]) for x in samples])     # the level drawn for each observation
mean, ds.noise_level_mean = ds.noise_level_mean, 0.0              # same observations without noise
clean = np.stack([ds[i]["observed"].numpy() for i in range(n)])
ds.noise_level_mean = mean
rel = (noisy - clean) / clean                                     # (n, N) relative noise

fig, axes = plt.subplots(1, 3, figsize=(15, 3.3))
axes[0].plot(wl, clean[0], lw=0.7, label="stored (clean)"); axes[0].plot(wl, noisy[0], lw=0.5, alpha=0.7, label="with noise")
axes[0].set_xlim(wl[0], wl[0] + (wl[-1] - wl[0]) / 10); axes[0].legend(); axes[0].set_title("zoom: clean vs noisy input")
axes[1].hist(levels, bins=20); axes[1].axvline(d.noise_level_mean, color="k", ls="--")
axes[1].set_title(f"noise level per observation: {levels.mean():.4f} ± {levels.std():.4f}\n(target {d.noise_level_mean} ± {d.noise_level_std})")
axes[2].hist((rel / levels[:, None]).ravel(), bins=80, density=True)
axes[2].set_title("(noisy - clean) / (clean x level): should be N(0, 1)")
plt.tight_layout(); plt.show()

# augmentation: the same observation in two repeats gets two different noise realizations
if dm.val_ds.repeats > 1:
    a, b = dm.val_ds[0]["observed"].numpy(), dm.val_ds[dm.val_ds._n_base()]["observed"].numpy()
    print(f"validation observation 0, repeat 1 vs 2: max |difference| = {np.abs(a - b).max():.4f} (different noise)")

## 4. Model

In [ ]:
from tellurics.scripts.train import build_model

module = build_model(m, t)            # LightningModule wrapping the transformer autoencoder
counts = module.model.parameter_counts()
for name, n in counts.items():
    print(f"  {name:20s} {n:>10,}")
print(f"  {'total':20s} {module.model.count_parameters():>10,}")

tm = module.model
print(f"tokens: {tm.n_patches} patches of {tm.patch_size} samples (+1 metadata token), "
      f"{m.num_latents} latent tokens x {m.latent_dim}")

batch = next(iter(dm.val_dataloader()))
print({k: tuple(v.shape) for k, v in batch.items() if torch.is_tensor(v)}, "metadata", tuple(batch["theta"]["metadata"].shape))
with torch.no_grad():
    out = module._run_estimator(batch)
print("prediction:", tuple(out.telluric.shape), " latent:", tuple(out.latent.shape))

plt.figure(figsize=(11, 3))
plt.plot(wl, batch["telluric"][0], lw=0.7, label="target")
plt.plot(wl, out.telluric[0], lw=0.9, label="untrained prediction (clear-sky start)")
plt.legend(); plt.title("before training"); plt.xlabel(f"wavelength [{dm.wavegrid.unit}]"); plt.show()

## 5. Training

A short run (`MAX_EPOCHS`; one epoch = every training observation x `noise_repeats`). Losses are recorded per epoch; the
best checkpoint (lowest `validation_loss`) is kept in `RUN_DIR`.

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint

class History(pl.Callback):
    """Collect the epoch-level metrics for plotting."""
    def __init__(self):
        self.train, self.val = [], []
    def on_validation_epoch_end(self, trainer, pl_module):
        if not trainer.sanity_checking:
            self.val.append(float(trainer.callback_metrics["validation_loss"]))
    def on_train_epoch_end(self, trainer, pl_module):
        self.train.append(float(trainer.callback_metrics["training_loss"]))

history = History()
checkpoint = ModelCheckpoint(dirpath=RUN_DIR / "checkpoints", monitor="validation_loss", mode="min",
                             filename="best-{epoch:03d}-{validation_loss:.5f}")
trainer = pl.Trainer(
    max_epochs=t.max_epochs, accelerator="cpu" if DEVICE == "cpu" else "auto",
    callbacks=[history, checkpoint], logger=False, enable_model_summary=False,
    gradient_clip_val=t.gradient_clip_val, log_every_n_steps=10,
)
trainer.fit(module, datamodule=dm)
print("best checkpoint:", checkpoint.best_model_path, " val loss:", float(checkpoint.best_model_score))

## 6. Loss curves

In [ ]:
epochs = np.arange(1, len(history.train) + 1)
plt.figure(figsize=(7, 3.5))
plt.semilogy(epochs, history.train, "o-", label="training")
plt.semilogy(epochs, history.val[: len(epochs)], "s-", label="validation")
plt.xlabel("epoch"); plt.ylabel("MSE (telluric spectrum)"); plt.legend(); plt.title("loss")
plt.show()

## 7. Evaluation on the test nights (best checkpoint)

In [ ]:
from tellurics.training.predict import load_trained_module, predict_loader

best = load_trained_module(checkpoint.best_model_path, map_location="cpu")
test_loader = dm.test_dataloader()
out = predict_loader(best, test_loader)
pred, observed = out["telluric_pred"], out["observed"]                    # (n, N); observed = noisy input
truth = np.concatenate([b["telluric"].numpy() for b in test_loader])
meta = np.concatenate([b["theta"]["metadata"].numpy() for b in test_loader])
res = pred - truth
print(f"test MSE {np.mean(res**2):.3e}   RMSE {np.sqrt(np.mean(res**2)):.3e}   max |res| {np.abs(res).max():.3e}")

# a few test observations: prediction vs truth, and residuals
rows = np.linspace(0, len(pred) - 1, 3).astype(int)
fig, axes = plt.subplots(len(rows), 1, figsize=(11, 2.4 * len(rows)), sharex=True)
for ax, i in zip(axes, rows):
    ax.plot(wl, truth[i], lw=0.8, label="true telluric")
    ax.plot(wl, pred[i], lw=0.8, ls="--", label="predicted")
    ax2 = ax.twinx(); ax2.plot(wl, res[i], lw=0.5, color="C3", alpha=0.6); ax2.set_ylabel("residual", color="C3")
    ax.set_ylabel("T"); ax.set_title(f"test observation {i}", fontsize=9)
axes[0].legend(loc="lower left"); axes[-1].set_xlabel(f"wavelength [{dm.wavegrid.unit}]")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
axes[0].plot(wl, np.sqrt(np.mean(res**2, axis=0)), lw=0.7)
axes[0].set_title("RMSE per wavelength"); axes[0].set_xlabel(f"wavelength [{dm.wavegrid.unit}]")
axes[1].hist(res.ravel(), bins=100, log=True); axes[1].set_title("residuals (all pixels)")
rmse_obs = np.sqrt(np.mean(res**2, axis=1))
for j, c in enumerate(d.metadata_columns[:2]):
    axes[2].scatter(meta[:, j], rmse_obs, s=6, alpha=0.6, label=c)
axes[2].set_xlabel("metadata (scaled)"); axes[2].set_ylabel("RMSE per observation"); axes[2].legend()
axes[2].set_title("error vs airmass / PWV")
plt.tight_layout(); plt.show()

### 7b. Telluric correction

Dividing the (noisy) observed flux by the predicted telluric spectrum removes the tellurics; dividing by the *true* telluric
spectrum shows what a perfect model would give.

In [ ]:
i = rows[len(rows) // 2]
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(wl, observed[i], lw=0.6, label="observed (input)")
axes[0].legend(loc="lower left"); axes[0].set_title(f"test observation {i}")
axes[1].plot(wl, observed[i] / truth[i], lw=0.6, label="observed / true telluric")
axes[1].plot(wl, observed[i] / pred[i], lw=0.6, alpha=0.8, label="observed / predicted telluric")
axes[1].legend(loc="lower left"); axes[1].set_xlabel(f"wavelength [{dm.wavegrid.unit}]")
plt.tight_layout(); plt.show()

## Summary

* data: single frames, simulated, stored observed flux (no stellar spectra), noise added in the pipeline, night-level split;
* model: transformer autoencoder, `(B, 2048) + metadata (B, 2) → (B, 2048)`;
* the best checkpoint is in `RUN_DIR/checkpoints/`.

For full runs use the training script (`python -m tellurics.scripts.train --config experiments/single_frame/experiment_transformer_ae.yaml`)
or the sweep (`python -m src.tellurics.scripts.sweep_transformer_ae`).